In [ ]:
import sys
from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import copy
import matplotlib.pyplot as plt

def find_project_root(start=None):
    if start is None:
        start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    return start


PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT))


In [ ]:
from src.io.pitch_io import load_pitch_file, load_preprocessed_pitch
from src.io.annotation_io import load_annotations
import settings as S

START_COL = "start_time_sec"
END_COL = "end_time_sec"


RAW_COL = S.PITCH_COL          # corba original per comparar
PITCH_COL = S.PITCH_COL        # o la columna que realment va usar el model (ex: "f0_Hz_savgol")
VALID_COL = "valid_for_pchip"

MAX_GROUP_LEN = 4
CONTEXT = 32                   # context a cada costat del grup
N_PLOTS = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



In [ ]:

# =========================================================
# CONFIG
# =========================================================



# =========================================================
# HELPERS
# =========================================================

def find_true_groups(mask: np.ndarray, max_len: int = 4):
    """
    Retorna una llista de tuples (start, end) per grups contigus True.
    'end' és exclusiu.
    Només conserva grups amb longitud <= max_len.
    """
    mask = np.asarray(mask, dtype=bool)
    groups = []
    n = len(mask)
    i = 0

    while i < n:
        if not mask[i]:
            i += 1
            continue

        j = i
        while j < n and mask[j]:
            j += 1

        if (j - i) <= max_len:
            groups.append((i, j))

        i = j

    return groups


def merge_groups(groups):
    """
    Fusiona grups solapats o adjacents.
    """
    if not groups:
        return []

    groups = sorted(groups, key=lambda x: x[0])
    merged = [groups[0]]

    for s, e in groups[1:]:
        ps, pe = merged[-1]
        if s <= pe:   # solapats o adjacents
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))

    return merged


def build_mask_from_groups(n, groups):
    mask = np.zeros(n, dtype=bool)
    for s, e in groups:
        mask[s:e] = True
    return mask


def get_window_bounds(start, end, n, context=32):
    ws = max(0, start - context)
    we = min(n, end + context)
    return ws, we


def safe_fill_for_model(x, gap_mask):
    """
    Valor provisional per donar entrada al model a les posicions emmascarades.
    Aquí usem interpolació lineal simple dins la finestra.
    Si no es pot, fem fallback a mediana local.
    """
    x = x.copy()
    idx = np.arange(len(x))
    valid = ~gap_mask & np.isfinite(x)

    if valid.sum() >= 2:
        x[gap_mask] = np.interp(idx[gap_mask], idx[valid], x[valid])
    elif valid.sum() == 1:
        x[gap_mask] = x[valid][0]
    else:
        x[gap_mask] = 0.0

    return x


# =========================================================
# MODEL INFERENCE
# =========================================================

def infer_window_with_model(model, x_window, gap_mask_window, device="cpu"):
    """
    Aquesta funció assumeix una interfície bastant típica:
    - entrada: [B, C, T]
    - C=2, on:
        canal 0 = corba amb el gap provisionalment omplert
        canal 1 = màscara del gap (1.0 on falta valor)
    - sortida: seqüència reconstruïda de longitud T

    Si el teu model té una altra signatura, adapta NOMÉS aquesta funció.
    """
    x_filled = safe_fill_for_model(x_window, gap_mask_window)

    x_tensor = torch.tensor(x_filled, dtype=torch.float32, device=device)[None, None, :]
    m_tensor = torch.tensor(gap_mask_window.astype(np.float32), dtype=torch.float32, device=device)[None, None, :]

    model_in = torch.cat([x_tensor, m_tensor], dim=1)   # [1, 2, T]

    model.eval()
    with torch.no_grad():
        y_hat = model(model_in)

    # Casos habituals
    if isinstance(y_hat, (tuple, list)):
        y_hat = y_hat[0]

    y_hat = y_hat.detach().cpu().numpy()

    # [1, 1, T] -> [T]
    if y_hat.ndim == 3:
        y_hat = y_hat[0, 0]
    elif y_hat.ndim == 2:
        y_hat = y_hat[0]

    return y_hat


# =========================================================
# CORE RECONSTRUCTION
# =========================================================

def reconstruct_short_groups_with_model(
    df_pitch: pd.DataFrame,
    df_aux: pd.DataFrame,
    model,
    pitch_col: str = PITCH_COL,
    raw_col: str = RAW_COL,
    valid_col: str = VALID_COL,
    max_group_len: int = 4,
    context: int = 32,
    device: str = "cpu",
):
    """
    Omple:
    1) grups curts de is_outlier == True
    2) grups curts de (f0_Hz == 0) & (valid_for_pchip == True)

    Retorna:
    - df_out amb columna pitch_reconstructed
    - groups_all: llista de grups processats
    - masks info
    """
    df_out = df_pitch.copy().reset_index(drop=True)
    df_aux = df_aux.copy().reset_index(drop=True)

    if len(df_out) != len(df_aux):
        raise ValueError(
            f"df_pitch i df_aux no tenen la mateixa longitud: {len(df_out)} vs {len(df_aux)}"
        )

    # -------- condició 1: outliers curts --------
    mask_outlier = df_aux["is_outlier"].fillna(False).to_numpy()
    groups_outlier = find_true_groups(mask_outlier, max_len=max_group_len)

    # -------- condició 2: zeros + valid_for_pchip curts --------
    mask_zero_valid = (
        (df_aux["f0_Hz"].fillna(0) == 0).to_numpy()
        &
        (df_aux["valid_for_pchip"].fillna(False)).to_numpy()
    )
    groups_zero_valid = find_true_groups(mask_zero_valid, max_len=max_group_len)

    # unió
    groups_all = merge_groups(groups_outlier + groups_zero_valid)

    original_curve = df_out[pitch_col].to_numpy(dtype=float)
    reconstructed = original_curve.copy()

    for start, end in groups_all:
        ws, we = get_window_bounds(start, end, n=len(df_out), context=context)

        x_window = reconstructed[ws:we].copy()
        gap_mask_window = np.zeros(len(x_window), dtype=bool)
        gap_mask_window[(start - ws):(end - ws)] = True

        y_hat_window = infer_window_with_model(
            model=model,
            x_window=x_window,
            gap_mask_window=gap_mask_window,
            device=device,
        )

        # només reescrivim el grup
        reconstructed[start:end] = y_hat_window[(start - ws):(end - ws)]

    df_out["pitch_reconstructed"] = reconstructed

    # màscares finals per debug
    df_out["mask_outlier_short"] = build_mask_from_groups(len(df_out), groups_outlier)
    df_out["mask_zero_valid_short"] = build_mask_from_groups(len(df_out), groups_zero_valid)
    df_out["mask_reconstructed_union"] = build_mask_from_groups(len(df_out), groups_all)

    return df_out, groups_all, groups_outlier, groups_zero_valid


# =========================================================
# PLOTS
# =========================================================

def plot_reconstruction_examples(
    df_out: pd.DataFrame,
    groups_all,
    raw_col: str = RAW_COL,
    recon_col: str = "pitch_reconstructed",
    n_plots: int = 8,
    context: int = 40,
    title_prefix: str = "",
):
    """
    Mostra fragments al voltant dels grups reconstruïts.
    """
    if len(groups_all) == 0:
        print("No hi ha grups per plotar.")
        return

    groups_to_plot = groups_all[:n_plots]
    n = len(df_out)

    raw = df_out[raw_col].to_numpy(dtype=float)
    recon = df_out[recon_col].to_numpy(dtype=float)

    for k, (start, end) in enumerate(groups_to_plot, 1):
        ws = max(0, start - context)
        we = min(n, end + context)
        x = np.arange(ws, we)

        plt.figure(figsize=(12, 4))
        plt.plot(x, raw[ws:we], label=raw_col, linewidth=1.8)
        plt.plot(x, recon[ws:we], label=recon_col, linewidth=1.8)

        plt.axvspan(start, end - 1, alpha=0.2)
        plt.title(f"{title_prefix} fragment {k} | group=({start}, {end}) len={end-start}")
        plt.xlabel("sample")
        plt.ylabel("Hz")
        plt.legend()
        plt.tight_layout()
        plt.show()


# =========================================================
# HIGH-LEVEL RUN PER RECORDING
# =========================================================

def run_reconstruction_for_recording(
    recording_id,
    tonic_hz,
    aux_pitch_path,
    model,
    root_dir,
    device="cpu",
    convert_to_cents=False,
):
    """
    Carrega dades, reconstrueix i plota.
    """
    df_pitch = load_preprocessed_pitch(
        recording_id=recording_id,
        root_dir=root_dir,
        tonic_hz=tonic_hz,
        convert_to_cents=convert_to_cents,
    )

    df_aux = load_pitch_file(file_path=aux_pitch_path)

    df_out, groups_all, groups_outlier, groups_zero_valid = reconstruct_short_groups_with_model(
        df_pitch=df_pitch,
        df_aux=df_aux,
        model=model,
        pitch_col=PITCH_COL,
        raw_col=RAW_COL,
        valid_col=VALID_COL,
        max_group_len=MAX_GROUP_LEN,
        context=CONTEXT,
        device=device,
    )

    print(f"recording_id={recording_id}")
    print(f"groups_outlier_short:    {len(groups_outlier)}")
    print(f"groups_zero_valid_short: {len(groups_zero_valid)}")
    print(f"groups_total_union:      {len(groups_all)}")

    plot_reconstruction_examples(
        df_out=df_out,
        groups_all=groups_all,
        raw_col=RAW_COL,
        recon_col="pitch_reconstructed",
        n_plots=N_PLOTS,
        context=40,
        title_prefix=f"{recording_id} | ",
    )

    return df_out, groups_all


# =========================================================
# EXEMPLE D'ÚS
# =========================================================

# 1) carrega el model entrenat "bo" de 3-4 samples
# checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)
# model = YourModelClass(...)
# model.load_state_dict(checkpoint["model_state_dict"])
# model.to(DEVICE)

# 2) executa per un recording
# df_out, groups_all = run_reconstruction_for_recording(
#     recording_id=recording_id,
#     tonic_hz=tonic_hz,
#     aux_pitch_path=aux_pitch_path,
#     model=model,
#     root_dir=S.DATA_INTERIM,
#     device=DEVICE,
#     convert_to_cents=False,   # perquè volem comparar en Hz
# )